# Tutorial 1: Learn about `requests`

Follow along with [this tutorial on RealPython](https://realpython.com/python-requests/). 
Complete the first 5 sections thoroughly (up to but not including User Other HTTP methods). Then jump to section Improve Performance to learn about some advanced tricks that you might need at some point when scraping websites that have a lot of content.

Use markdown headings as appropriate to enumerate sections.

**Author**: Serena Day   
**Created on:** Sep 9, 2026

**Table of Contents**

1. [Make a GET Request](#sec1)
2. [Inspect the Response](#sec2)
3. [Add String Query Parameters](#sec3)
4. [Customize Request Headers](#sec4)
5. [Improve Performance](#sec5)

In [1]:
import requests

<h2 id="sec2"> Make a GET Request </h2>

requests.get() --> retrieves data from specified resource

In [2]:
response = requests.get("https://api.github.com")

<h2 id="sec1"> Inspect the Response </h2>

response.status_code --> 200 (success), 404 (not found)

In [3]:
if response.status_code == 200:
    print("Success!")
elif response.status_code == 404:
    print("Note Found.")

# same as
if response:
    print("Success!")
else:
    raise Exception(f"Non-success status code: {response.status_code}")


Success!
Success!


response.raise_for_status() --> requests will raise an HTTPError for status codes between 400 and 600

In [4]:
from requests.exceptions import HTTPError 

URLS = ["https://api.github.com", "https://api.github.com/invalid"]

for url in URLS:
    try: 
        response = requests.get(url)
        response.raise_for_status()
    except HTTPError as http_err:
        print(f"HTTP error occurred: {http_err}")
    except Exception as err:
        print(f"Other error occurred: {err}")
    else:
        print("Success!")

Success!
HTTP error occurred: 404 Client Error: Not Found for url: https://api.github.com/invalid


response.content --> seeing the response content in bytes

In [5]:
response = requests.get("https://api.github.com")
print(response.content)
type(response.content)

b'{\n  "current_user_url": "https://api.github.com/user",\n  "current_user_authorizations_html_url": "https://github.com/settings/connections/applications{/client_id}",\n  "authorizations_url": "https://api.github.com/authorizations",\n  "code_search_url": "https://api.github.com/search/code?q={query}{&page,per_page,sort,order}",\n  "commit_search_url": "https://api.github.com/search/commits?q={query}{&page,per_page,sort,order}",\n  "emails_url": "https://api.github.com/user/emails",\n  "emojis_url": "https://api.github.com/emojis",\n  "events_url": "https://api.github.com/events",\n  "feeds_url": "https://api.github.com/feeds",\n  "followers_url": "https://api.github.com/user/followers",\n  "following_url": "https://api.github.com/user/following{/target}",\n  "gists_url": "https://api.github.com/gists{/gist_id}",\n  "hub_url": "https://api.github.com/hub",\n  "issue_search_url": "https://api.github.com/search/issues?q={query}{&page,per_page,sort,order}",\n  "issues_url": "https://api.

bytes

response.text --> seeing response in UTF-8

In [6]:
response.encoding = "utf-8"  # Optional: Requests infers this
print(response.text)
type(response.text)

{
  "current_user_url": "https://api.github.com/user",
  "current_user_authorizations_html_url": "https://github.com/settings/connections/applications{/client_id}",
  "authorizations_url": "https://api.github.com/authorizations",
  "code_search_url": "https://api.github.com/search/code?q={query}{&page,per_page,sort,order}",
  "commit_search_url": "https://api.github.com/search/commits?q={query}{&page,per_page,sort,order}",
  "emails_url": "https://api.github.com/user/emails",
  "emojis_url": "https://api.github.com/emojis",
  "events_url": "https://api.github.com/events",
  "feeds_url": "https://api.github.com/feeds",
  "followers_url": "https://api.github.com/user/followers",
  "following_url": "https://api.github.com/user/following{/target}",
  "gists_url": "https://api.github.com/gists{/gist_id}",
  "hub_url": "https://api.github.com/hub",
  "issue_search_url": "https://api.github.com/search/issues?q={query}{&page,per_page,sort,order}",
  "issues_url": "https://api.github.com/issues

str

response.json() --> serialized JSON

In [7]:
response.json()
type(response.json())

dict

can access values in object by key

In [8]:
response_dict = response.json()
response_dict["emojis_url"]

'https://api.github.com/emojis'

view response headers  
response.headers --> viewing headers

In [9]:
response = requests.get("https://api.github.com")
response.headers

# seeing content type of response payload
response.headers["Content-Type"] # case insensitive

'application/json; charset=utf-8'

<h2 id="sec3"> Add Query String Parameters </h2>
customizing GET requests  
requests.get("link",params={...})

In [10]:
response = requests.get(
    "https://api.github.com/search/repositories",
    params={"q": "language:python", "sort": "stars", "order": "desc"},
    )

json_response = response.json()
popular_repositories = json_response["items"]

for repo in popular_repositories[:3]:
    print(f"Name: {repo['name']}")
    print(f"Description: {repo['description']}")
    print(f"Stars: {repo['stargazers_count']}\n")

Name: public-apis
Description: A collective list of free APIs
Stars: 478694

Name: free-programming-books
Description: :books: Freely available programming books
Stars: 396470

Name: system-design-primer
Description: Learn how to design large-scale systems. Prep for the system design interview.  Includes Anki flashcards.
Stars: 369310



In [12]:
# params in get() can also be list of tuples
requests.get(
    "https://api.github.com/search/repositories",
    [("q", "language:python"), ("sort", "stars"), ("order", "desc")]
)

# passing values as bytes
requests.get("https://api.github.com/search/repositories",
             params=b"q=language:python&sort=stars&order=desc")

<Response [200]>

<h2 id="sec4"> Customize Request Headers </h2>
pass a dictionary of HTTP headers to get() using headers parameter

In [18]:
response = requests.get(
    "https://api.github.com/search/repositories",
    params={"q": '"real python"'},
    headers={"Accept": "application/vnd.github.text-match+json"}
)

json_response = response.json()
first_repository = json_response["items"][0]
print(first_repository["text_matches"][0]["matches"])


[{'text': 'Real Python', 'indices': [23, 34]}]


<h2 id="sec5"> Improve Performance </h2>
#### Set Request Timeouts
its good to specify a timeout duration to prevent waiting indefinitely

In [21]:
requests.get("https://api.github.com", timeout=1)
#requests.get("https://api.github.com", timeout=0.01)

<Response [200]>

In [22]:
# Passing a tuple with (connect timeout, read timeout)
requests.get("https://api.github.com", timeout=(3.05, 5))

<Response [200]>

In [23]:
from requests.exceptions import Timeout

try:
    response = requests.get("https://api.github.com", timeout=(3.05, 5))
except Timeout:
    print("the request has timed out")
else:
    print("the request did not time out")

the request did not time out


#### Reuse Connections with Session Objects
fine tune control over how requests are made- persist parameters across requests

In [27]:
from requests.auth import AuthBase
from custom_token_auth import TokenAuth

TOKEN = "<YOUR_GITHUB_PA_TOKEN>"

with requests.Session() as session:
    session.auth = TokenAuth(TOKEN)

    first_response = session.get("https://api.github.com/user")
    second_response = session.get("https://api.github.com/user")

print(first_response.headers)
print(second_response.json())


{'Content-Type': 'application/json; charset=utf-8', 'X-GitHub-Media-Type': 'github.v3; format=json', 'Access-Control-Expose-Headers': 'ETag, Link, Location, Retry-After, X-GitHub-OTP, X-RateLimit-Limit, X-RateLimit-Remaining, X-RateLimit-Used, X-RateLimit-Resource, X-RateLimit-Reset, X-OAuth-Scopes, X-Accepted-OAuth-Scopes, X-Poll-Interval, X-GitHub-Media-Type, X-GitHub-SSO, X-GitHub-Request-Id, X-GitHub-Edge-Region, Deprecation, Sunset', 'Access-Control-Allow-Origin': '*', 'Strict-Transport-Security': 'max-age=31536000; includeSubdomains; preload', 'X-Frame-Options': 'deny', 'X-Content-Type-Options': 'nosniff', 'X-XSS-Protection': '0', 'Referrer-Policy': 'origin-when-cross-origin, strict-origin-when-cross-origin', 'Content-Security-Policy': "default-src 'none'", 'Vary': 'Accept-Encoding, Accept, X-Requested-With', 'Server': 'github.com', 'X-GitHub-Request-Id': 'C275:92563:17E96EF:4C466C6:6AA38444', 'x-github-edge-region': 'iad', 'Date': 'Fri, 11 Sep 2026 04:32:04 GMT', 'connection': '

#### Retry Failed Requests
must build a transport adaptor, which allows you to define a set of configurations for each service you interact with

In [29]:
import requests
from requests.adapters import HTTPAdapter
from requests.exceptions import RetryError
from urllib3.util.retry import Retry

retry_strategy = Retry(
    total = 2,
    status_forcelist = [429,500,502,503,504]
)
github_adapter = HTTPAdapter(max_retries=retry_strategy)

# new session
with requests.Session() as session:
    session.mount("https://api.github.com", github_adapter)
    try:
        response = session.get("https://api.github.com/")
    except RetryError as err:
        print(f"Error: {err}")